# Pillar 2z — Policy distillation on V12 (Colab H100)

Train pillar2z by warm-starting from pillar2y2_epoch_40 on the V12 corpus (9.77M states / 5.8GB: 450 self-play games + ~9700 crisis recovery + ~9700 prevention replays).

**Per Pillar 2u (HISTORY 110+):** policy-only training, `val_weight=0`. The model has no value head; we don't predict score. Each state's target is the MCTS visit distribution from a 400-sim NN-MCTS search.

**Per Phase 3R (HISTORY 133-136):** the player that generated V12 was pillar2y2 + value_head_v11 + q=2.0 (Phase 3R: mean 13,476 / median 16,427 cap-pinned).

**Hyperparameters — same as the 2y recipe:**
- batch 32768, lr 3e-4, warmup 2 epochs (V11/2y A/B winner)
- AMP + torch.compile
- Warm-start from pillar2y2_epoch_40

**Epoch budget — be ready to extend.** Trajectory has been monotonically up: 2x:10 → 2y:20 → 2y2:40, each extension because the prior was still learning. V12 is a richer corpus than V11, so 40 may again be too few. Plan: train 40, eyeball val_loss, run another 20 with `--resume` if still falling.

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_selfplay_train.tar.gz` — code
2. `v12_pillar2z.pt` — V12 training tensor (5.8 GB; built locally on M5)
3. `pillar2y2_epoch_40.pt` — warm-start checkpoint (~143 MB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
t0 = time.time()
!cp {DRIVE}/v12_pillar2z.pt /content/alphatrain/data/v12_pillar2z.pt
print(f'Tensor: {os.path.getsize("/content/alphatrain/data/v12_pillar2z.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!cp {DRIVE}/pillar2y2_epoch_40.pt /content/alphatrain/data/pillar2y2_epoch_40.pt
print(f'Warm-start ckpt: {os.path.getsize("/content/alphatrain/data/pillar2y2_epoch_40.pt")/1e6:.0f} MB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
EPOCHS = 40
SAVE_DIR = '/content/checkpoints/pillar2z'
DRIVE_SAVE_DIR = f'{DRIVE}/checkpoints_pillar2z'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train \
    --tensor-file alphatrain/data/v12_pillar2z.pt \
    --amp --compile \
    --resume alphatrain/data/pillar2y2_epoch_40.pt --warm-start \
    --policy-only \
    --epochs {EPOCHS} --batch-size 32768 --lr 3e-4 --warmup-epochs 2 \
    --copy-to {DRIVE_SAVE_DIR} \
    --save-dir {SAVE_DIR}

In [ ]:
# Copy ALL epoch checkpoints to Drive (mirror 2y notebook pattern,
# in case --copy-to only copies best.pt and you want every epoch).
import shutil, os, glob
for f in sorted(glob.glob(f'{SAVE_DIR}/epoch_*.pt')):
    dst = f'{DRIVE_SAVE_DIR}/pillar2z_{os.path.basename(f)}'
    if not os.path.exists(dst):
        shutil.copy(f, dst)
        print(f'  copied {os.path.basename(f)}')

## Notes

- **Wall time on H100:** ~1-2 min/epoch at batch 32768 → ~40-80 min for 40 epochs.
- **Resume to extend:** if val_loss still falling at 40, run another 20 with `--resume {DRIVE_SAVE_DIR}/best.pt` (drop `--warm-start` so optimizer continues).
- **Per-epoch save:** the second cell mirrors the 2y notebook's manual epoch-copy pattern in case `--copy-to` only saves `best.pt`.
- **After training:** download `best.pt` (or epoch_N.pt) from Drive. Eval locally:
  ```
  python -m alphatrain.scripts.eval_parallel \
    --model alphatrain/data/pillar2z_best.pt \
    --value-head-path alphatrain/data/value_head_v11.pt \
    --simulations 400 --q-weight 2.0 --workers 16 --device mps \
    --max-turns 8000 --mcts-only --early-stop \
    --seeds 0 1 2 3 ... 49
  ```
  Compare to pillar2y2 baseline (mean 13,476 / median 16,427 / %≥15K 69%). Watch policy-only mean too (currently 5,586 — should improve materially).